# Example: High resolution static solves

This notebook showcases how to leverage several optional features of FreeGSNKE in order to carry out more expensive static plasma calculations than usually possible.

The resolution of of the grid is generally limited by two factors: the amount of memory demanded from the system (hard limit), and the execution time required for the calculation to compute (soft limit). The following are the optional features offered by FreeGSNKE to address these issues, with some considerations the user may want to take into account before using them.

- *Multithreading* in critical computations
    - Can substantially **reduce execution time** in static solves
    - May **hamper performance** if used on **small grids**
    - On **personal computers** performance **may vary drastically** due to other running processes
- *On-the-fly computation* of the boundary Green's function
    - Significantly **reduces memory** required, allowing to run larger systems
    - Noticeably **increases execution time** due to recalculation
    - Reduction in memory use **can be tuned**
    - Strongly **discouraged** for evolutive solves
- Use of frequency-based solver
    - **Faster** than the default sparse solver
    - Uses **less memory** than the default sparse solver
    - Is **less accurate** ($O(\delta^2)$) than the default ($O(\delta^4)$)

## Import packages

In [ ]:
import pickle
import matplotlib.pyplot as plt
import numpy as np
from time import perf_counter


## Set up the STEP case


In [ ]:
from freegsnke import build_machine

tokamak = build_machine.tokamak(
    active_coils_path=f"../machine_configs/OpenSTEP/OpenSTEP_simple_active_coils.pickle",
    limiter_path=f"../machine_configs/OpenSTEP/OpenSTEP_simple_limiter.pickle",
    wall_path=f"../machine_configs/OpenSTEP/OpenSTEP_simple_limiter.pickle",
)

# These currents are from the FIESTA MATLAB file
currentvec = np.array([-4790870.39240653,  5155466.64491291,  2254704.04332795,
        5458947.31403196, -7854367.88994826,  5100000.        ,
       -2274646.53005716])

from scipy.interpolate import RegularGridInterpolator
from freegsnke.equilibrium import PsiGuessParabolicJtor

# We can use the provided data for a very close initial guess
with open('../machine_configs/OpenSTEP/OpenSTEP_psi_data.pickle', 'rb') as f:
    psi_data = pickle.load(f)

psiR=psi_data['R'][:,0]
psiZ=psi_data['Z'][0,:]
fInterp = RegularGridInterpolator((psiR,psiZ), psi_data['psi'],method='cubic',bounds_error=False, fill_value=None)


# Or as will be done for most of this example, we can use an initial guess obtained with the following utility function
psi_guess_factory = PsiGuessParabolicJtor(
    h = 4.0,
    k = 0.0,     # centre of ellipse (Z)
    a = 1.0,     # roughly the minor radius of the plasma
    b = 1.0*5, # roughly the minor radius times the elongation of expected plasma
    Ip = 2.2767e7, # plasma current (to scale the guess)
    psi_scale = 1.0   # might need to vary this default
)

# plot the machine
fig1, ax1 = plt.subplots(1, 1, figsize=(3, 8), dpi=80)
ax1.grid(zorder=0, alpha=0.75)
ax1.set_aspect('equal')
tokamak.plot(axis=ax1,show=False)
ax1.fill(tokamak.wall.R, tokamak.wall.Z, color='k', linewidth=1.2, facecolor='w', zorder=0) 
ax1.set_xlabel("Major radius [m]")
ax1.set_ylabel("Height [m]")
plt.show()

# some array to store results
resolutions = []
psi_results = []
x_points = []
o_points = []
jtor_results = []



## Calculating an equilibrium on a small grid using the defaults

Just to exemplify the regular workflow, we will show a solve on a small grid.

### Create the equilibrium with the chosen resolution

In [ ]:
from freegsnke import equilibrium_update
from freegsnke.jtor_update import GeneralPprimeFFprime

nr,nz=65,129

eq = equilibrium_update.Equilibrium(
    tokamak=tokamak,      # provide tokamak object
    Rmin=0.5, Rmax=9.0,   # radial range
    Zmin=-10.0, Zmax=10.0,  # vertical range
    nx=nr,                # number of grid points in the radial direction (needs to be of the form (2**n + 1) with n being an integer)
    ny=nz,               # number of grid points in the vertical direction (needs to be of the form (2**n + 1) with n being an integer)
)

eq.tokamak.set_all_coil_currents(currentvec)

# create the initial guess
_plasma_psi = psi_guess_factory(eq=eq) # keeping this for plotting later
eq.plasma_psi = _plasma_psi

profiles = GeneralPprimeFFprime(
    eq=eq,
    Ip=2.2767e7,
    fvac=0.5,
    psi_n=np.linspace(0,1,256),
    pprime_data=psi_data['pprime'],
    ffprime_data=psi_data['ffprime']
)

### Solve for the created equilibrium

In [ ]:
from freegsnke import GSstaticsolver



GSStaticSolver = GSstaticsolver.NKGSsolver(
    eq,
)

GSStaticSolver.solve(
    eq=eq,
    profiles=profiles,
    constrain=None,
    target_relative_tolerance=1e-10,
    force_up_down_symmetric=True,
    verbose=False
)

# save the results
resolutions.append((nr,nz))
psi_results.append(eq.psi())
x_points.append(profiles.xpt)
o_points.append(profiles.opt)
jtor_results.append(GSStaticSolver.jtor)


# Plot initial guess
fig0, (ax01,ax02) = plt.subplots(1, 2, figsize=(8, 8), dpi=80)
ax01.grid(zorder=0, alpha=0.75)
ax01.set_aspect('equal')
eq.tokamak.plot(axis=ax01,show=False)
ax01.plot(tokamak.wall.R, tokamak.wall.Z, 'k')
ax01.contour(eq.R, eq.Z, _plasma_psi)
ax01.set_title("Initial guess")

# plot the equilibrium
plt.ion()
ax02.grid(zorder=0, alpha=0.75)
ax02.set_aspect('equal')
eq.tokamak.plot(axis=ax02,show=False)                                                          # plots the active coils and passive structures
ax02.fill(tokamak.wall.R, tokamak.wall.Z, color='k', linewidth=1.2, facecolor='w', zorder=0)   # plots the limiter
eq.plot(axis=ax02,show=False)
ax02.set_title("Resulting equilibrium")
# ax02.contour(eq.R, eq.Z, eq.psi(), levels=np.linspace(np.amin(eq.psi()), np.amax(eq.psi()), 35))
plt.tight_layout()
# plt.legend()

## Setting the number of threads

Now we will run on a slightly larger grid and see what impact we get from using multiple threads.

We will set the threadpool size with the utility function `set_num_threads()` in `freegs4e.parallel`; this affects only the number of threads used by certain FreeGSNKE functions: it does not affect OpenMP threadpools like those used by `numpy` and `scipy`. It is possible to control all of these together by setting the environment variable `OMP_NUM_THREADS` before running a script.

Below, parallelism is being leveraged by: 1) the equilibrium initialization `equilibrium_update.Equilibrium()`, and 2) the solver initialization and execution `GSstaticsolver.NKGSsolver()`/`GSStaticSolver.solve()`: the number of threads should be set before calling those functions. Given that these two workloads are different, users may want to set different pool sizes for each of them.

#### (caching Green's function)

The testcase below also shows how to use *on-the-fly computation* of the Green's function. For a testcase this size it is not really necessary, but the additional computation will help see the effects of multithreading more clearly (execution time results on personal computers can be very noisy, especially when using many threads). Further down this notebook there is a more in-depth explanation of this tool.

***Warning:*** *the following cell is, by design, computationally expensive. It can take several minutes to complete*

In [ ]:
from freegsnke.parallel import set_num_threads

nr,nz=129,257
num_threads = [1,2,4]

set_num_threads(1) # sets threads to 1 (the default)

for nt in num_threads:
    t0 = perf_counter()

    set_num_threads(2) # setting 2 threads for equilibrium initialization
    eq = equilibrium_update.Equilibrium(
        tokamak=tokamak,      # provide tokamak object
        Rmin=0.5, Rmax=9.0,   # radial range
        Zmin=-10.0, Zmax=10.0,  # vertical range
        nx=nr,                # number of grid points in the radial direction (needs to be of the form (2**n + 1) with n being an integer)
        ny=nz,               # number of grid points in the vertical direction (needs to be of the form (2**n + 1) with n being an integer)
    )
    eq.tokamak.set_all_coil_currents(currentvec)

    eq.plasma_psi = psi_guess_factory(eq=eq)

    profiles = GeneralPprimeFFprime(
        eq=eq,
        Ip=2.2767e7,
        fvac=0.5,
        psi_n=np.linspace(0,1,256),
        pprime_data=psi_data['pprime'],
        ffprime_data=psi_data['ffprime']
    )

    t1 = perf_counter()

    set_num_threads(nt) # set number of threads for the solver
    GSStaticSolver = GSstaticsolver.NKGSsolver(
        eq,
        cache_greens=False, # COMPUTE ON-THE-FLY
    )

    GSStaticSolver.solve(
        eq=eq,
        profiles=profiles,
        constrain=None,
        target_relative_tolerance=1e-6,
        force_up_down_symmetric=True,
        verbose=False,
    )

    t2 = perf_counter()
    print(f"Solved in {t2-t1}s with {nt} threads (total {t2-t0}s)\n")

# save the results
resolutions.append((nr,nz))
psi_results.append(eq.psi())
x_points.append(profiles.xpt)
o_points.append(profiles.opt)
jtor_results.append(GSStaticSolver.jtor)

### Regarding results when using different numbers of threads

Note above that the number of iterations is consistent between runs: FreeGSNKE's implementation of threading for static solves is *fully deterministic*: numerical results are exactly replicated independently of the number of threads. Nonetheless, the same is not true of the parallel backends used by `numpy`, so results will present differences if parallelism is varied with environment variables, but not if controlled with the `freegsnke.parallel` API.

## Using the frequency-based solver

Now we explore the impact of using a different solver backend. The frequency-based "DST" solver can be selected with the optional argument `solver_type` when initializing the static solver `GSstaticsolver.NKGSsolver()`. The "DST" solver is less accurate than the default "LUsparse", but it can be up to an order of magnitude faster, which makes it particularly useful on large grids and for evolutive calculations.

Unlike all other features presented here, this one causes numerical differences, which will cause a change in the number of iterations executed with each method (and so muddy the comparison). Given that here we are primarily concerned about execution time, we set `max_solving_iterations` to a low value to ensure the speed comparison is fair.

***Warning:*** *the following cell is, by design, computationally expensive. It can take several minutes to complete*

In [ ]:
nr,nz=257,513
solver_options = ['LUsparse','DST'] # LUsparse is the default
set_num_threads(1)

for solver_type in solver_options:

    t0 = perf_counter()

    eq = equilibrium_update.Equilibrium(
        tokamak=tokamak,      # provide tokamak object
        Rmin=0.5, Rmax=9.0,   # radial range
        Zmin=-10.0, Zmax=10.0,  # vertical range
        nx=nr,                # number of grid points in the radial direction (needs to be of the form (2**n + 1) with n being an integer)
        ny=nz,               # number of grid points in the vertical direction (needs to be of the form (2**n + 1) with n being an integer)
    )
    
    eq.tokamak.set_all_coil_currents(currentvec)

    eq.plasma_psi = psi_guess_factory(eq=eq)

    profiles = GeneralPprimeFFprime(
        eq=eq,
        Ip=2.2767e7,
        fvac=0.5,
        psi_n=np.linspace(0,1,256),
        pprime_data=psi_data['pprime'],
        ffprime_data=psi_data['ffprime']
    )

    GSStaticSolver = GSstaticsolver.NKGSsolver(
        eq,
        solver_type=solver_type, # HERE WE SET THE SOLVER
    )


    t1 = perf_counter()

    GSStaticSolver.solve(
        eq=eq,
        profiles=profiles,
        constrain=None,
        target_relative_tolerance=1e-6,
        max_solving_iterations=40,
        force_up_down_symmetric=True,
        verbose=False
    )

    
    t2 = perf_counter()
    print(f"Solved in {t2-t1}s with {solver_type} solver (total {t2-t0}s)\n")

# # save the results
resolutions.append((nr,nz))
psi_results.append(eq.psi())
x_points.append(profiles.xpt)
o_points.append(profiles.opt)
jtor_results.append(GSStaticSolver.jtor)

## Leveraging on-the-fly computation to enable larger grids

As shown above, passing the argument `cache_greens=False` during solver initialization (`GSstaticsolver.NKGSsolver()`) activates *on-the-fly computation*. This will skip the computation of the Green's function during initialization, but will instead compute it from scratch on every iteration of the solve stage.

Not having to store the whole array simultaneously means a very significant (roughly 3-5 times) reduction in memory use. Nonetheless, it also carries an execution time penalty (up to 10x during testing). Here we make up for some of that lost performance by applying all the techniques presented in this notebook.

The flag `max_batch_size` can also be used to allow for a further decrease in memory, by reducing the size of the portion of the Green's function that can be held in memory at a time. We use it below as an example, but at this resolution it will not carry any effect (its main intended use is for very resolved grids on HPC systems). A good approximation is that the default `2000_000_000` maximum batch size puts an upper ceiling of ~64GB, so here we are setting an upper limit of 13GB, significantly higher than what we expect for this grid. This value can be reduced if needed to fit the computer you are using.

To keep down the execution time for this example, we will start from a closer guess, using the psi data we loaded previously for OpenSTEP.

***Warning:*** *the following cell is, by design, computationally expensive. It can take several minutes to complete*

In [ ]:
# Increase the resolution and rebuild equilibrium
nr,nz=513,1025 # around the maximum achievable on a personal computer
t0 = perf_counter()

set_num_threads(4)
eq = equilibrium_update.Equilibrium(
    tokamak=tokamak,      # provide tokamak object
    Rmin=0.5, Rmax=9.0,   # radial range
    Zmin=-10.0, Zmax=10.0,  # vertical range
    nx=nr,                # number of grid points in the radial direction (needs to be of the form (2**n + 1) with n being an integer)
    ny=nz,               # number of grid points in the vertical direction (needs to be of the form (2**n + 1) with n being an integer)
)
eq.tokamak.set_all_coil_currents(currentvec)

init_psi = -fInterp((eq.R,eq.Z))/(2*np.pi)
tokamak_psi = eq.tokamak.calcPsiFromGreens(pgreen=eq._pgreen)
eq.plasma_psi = init_psi - tokamak_psi

profiles = GeneralPprimeFFprime(
        eq=eq,
        Ip=2.2767e7,
        fvac=0.5,
        psi_n=np.linspace(0,1,256),
        pprime_data=psi_data['pprime'],
        ffprime_data=psi_data['ffprime']
)

t1 = perf_counter()
print(f"Equilibrium (and profiles) generated in {t1-t0}s",flush=True)

set_num_threads(8)
GSStaticSolver = GSstaticsolver.NKGSsolver(
    eq,
    solver_type='DST',
    cache_greens=False,
    max_batch_size=400_000_000,
)

GSStaticSolver.solve(
    eq=eq,
    profiles=profiles,
    constrain=None,
    target_relative_tolerance=1e-6,
    force_up_down_symmetric=True,
    verbose=True
)

t2 = perf_counter()
print(f"Equilibrium solved in {t2-t1}s")

# save the results
resolutions.append((nr,nz))
psi_results.append(eq.psi())
x_points.append(profiles.xpt)
o_points.append(profiles.opt)
jtor_results.append(GSStaticSolver.jtor)

### When should *on-the-fly* be used?

This feature is intended only as a workaround to the hard limit of available system RAM. It should only be activated when a grid is otherwise too large to be simulated &mdash; whether this is the case will be made evident by a `numpy` memory error or an interpreter crash at the point of initializing the solver (the call to `GSstaticsolver.NKGSsolver()`).

Additionally, `cache_greens=False` should be **avoided completely** when the intention is to perform an evolutive solve: in this case, the performance cost will become prohibitively expensive and the memory savings will be less significant. There are other tools for reducing memory use in evolutive solves, some of which are explained in **example 05a**.

## Comparison of results

At this point we have saved results for 4 different grid resolutions, so we can now compare them, in order to see the effect of using more resolved grids.

### Shape of the plasma

In [ ]:

num_figs = len(resolutions)+1
fig1, axes = plt.subplots(1, num_figs, figsize=(3.5*num_figs*1.2, 7), dpi=80)

# plot initial guess
ax0 = axes[0]
ax0.grid(zorder=0, alpha=0.75)
ax0.set_aspect('equal')
eq.tokamak.plot(axis=ax01,show=False)
ax0.plot(tokamak.wall.R, tokamak.wall.Z, 'k')
R, Z = np.meshgrid(
    np.linspace(eq.Rmin,eq.Rmax,_plasma_psi.shape[0]),
    np.linspace(eq.Zmin,eq.Zmax,_plasma_psi.shape[1]),
    indexing="ij"
)
ax0.contour(R, Z, _plasma_psi)
ax0.set_title("Initial guess")

for i in range(len(resolutions)):

    nr,nz = resolutions[i]
    axes[i+1].set_title(f"${nr}\\times{nz}$")
    axes[i+1].grid(zorder=0, alpha=0.75)
    axes[i+1].set_aspect('equal')
    eq.tokamak.plot(axis=axes[i+1],show=False)                                                          # plots the active coils and passive structures
    axes[i+1].fill(tokamak.wall.R, tokamak.wall.Z, color='k', linewidth=1.2, facecolor='w', zorder=0)   # plots the limiter
    R, Z = np.meshgrid(
        np.linspace(eq.Rmin,eq.Rmax,nr),
        np.linspace(eq.Zmin,eq.Zmax,nz),
        indexing="ij"
    )
    axes[i+1].contour(R, Z, psi_results[i],levels=np.linspace(np.amin(psi_results[i]), np.amax(psi_results[i]), 35))

    axes[i+1].contour(R,Z,psi_results[i],levels=[x_points[i][0][2]],colors='r')
    for r, z, _ in x_points[i]:
        axes[i+1].plot(r, z, "rx", markersize=9)
        axes[i+1].plot(x_points[i][0][0], x_points[i][0][1], "rx", markersize=9, markeredgewidth=2.5)
    for r, z, _ in o_points[i]:
        axes[i+1].plot(r, z, "g2", markersize=9)

    axes[i+1].set_yticklabels([])
    axes[i+1].set_ylabel("")



### Jtor magnitude

We now plot jtor over a few horizontal lines. These results can give indication of what minimum resolution may be necessary to achieve convergence, while also showing how more refined grids allow to resolve smaller, more detailed, features.

In [ ]:
def extract_jtor(jtor,z_vals):
    midline = jtor.shape[-1]//2
    return tuple(jtor[:,midline + int(z_val/20.*jtor.shape[-1])] for z_val in z_vals)


num_figs = len(resolutions)
fig2, axes2 = plt.subplots(1, num_figs, figsize=(3.5*num_figs*1.2, 7), dpi=80)
max_jtor = max([jtor.max() for jtor in jtor_results])

for i in range(len(resolutions)):

    nr,nz = resolutions[i]
    axes2[i].set_title(f"${nr}\\times{nz}$")
    img = axes2[i].imshow(jtor_results[i].T,vmin=0,vmax=max_jtor,extent=(0.5,9.,-10,10),cmap='inferno')
    axes2[i].fill(tokamak.wall.R, tokamak.wall.Z, color='b', linewidth=1.2, facecolor=('w',0), zorder=0)   # plots the limiter
    axes2[i].grid(zorder=0, alpha=0.75)
    eq.tokamak.plot(axis=axes2[i],show=False)                                                          # plots the active coils and passive structures
    if i!=0:
        axes2[i].set_yticklabels([])
        axes2[i].set_ylabel("")
        img0 = img

fig2.colorbar(img0,ax=axes2,orientation="vertical",fraction=0.1)

z_vals = [6,5,4,0]
num_figs = len(z_vals)
fig1, axes = plt.subplots(num_figs, 1, figsize=(12, 3*num_figs), dpi=80)

for jtor_field,(nr,nz) in zip(jtor_results,resolutions):
    jtor_curves = extract_jtor(jtor_field,z_vals)
    n_curves = len(jtor_curves)
    for i in range(n_curves):
        jtor = jtor_curves[i]
        axes[i].plot(np.linspace(0.5,9.,len(jtor)),jtor,label=f"${nr}\\times{nz}$")
        axes[i].set_ylim(0,max(1e6,jtor.max()*1.05))

for i in range(num_figs):
    axes[i].set_ylabel(f"$z={z_vals[i]}$")
    axes[i].grid()
    axes[i].set_xlim(1.5,6)

axes[0].legend(title='Resolution')
axes[num_figs//2].set_ylabel("Jtor\n"+axes[num_figs//2].get_ylabel())
axes[-1].set_xlabel("Major radius [m]")
plt.show()
